## Let's get the domains for the entire tree and plot them

In [1]:
from collections import defaultdict  # LP: required for parsing the domain data
from pathlib import Path

# We'll need ETE3 to parse and render the tree
from ete3 import Tree, TreeStyle, NodeStyle, TextFace, CircleFace, faces, SeqMotifFace
from tqdm.notebook import tqdm
import pandas as pd

In [2]:
# File locations and other constants
treepath = Path("TBE_named_tree_manual.new")  # transporter family tree, Newick format
annopath = Path("annotations.csv")     # annotations for some family members


In [3]:
# Load the tree and, for convenience, root it to the midpoint
# We have these as functions so that we can annotate/render a clean tree each time
def load_tree():
    tree = Tree(str(treepath))
    # midpoint = tree.get_midpoint_outgroup()  # Calculate the midpoint node
    # tree.set_outgroup(midpoint)# and set it as tree outgroup
    # tree.ladderize()
    return tree

# Load the annotation
def load_annotation():
    anno = pd.read_csv(annopath)
    return anno


In [4]:
load_annotation()  # Load the unaltered annotation as it is in the file

,id,kingdom,match
0,A0A556RIY8,Bacteria,NaN
1,K8W564,Bacteria,NaN
2,A0AA87CRL5,Bacteria,NaN
3,B6XEC6,Bacteria,NaN
4,K8X885,Bacteria,NaN
...,...,...,...
1671,A0A7R6P982,Bacteria,NaN
1672,A0A7R6PDR3,Bacteria,NaN
1673,A0A7U8C906,Bacteria,NaN
1674,A0A2G6JCW0,Bacteria,NaN


In [5]:
# The function lets us return the tree and its corresponding annotation,
# where the leaves of this specific tree object are present in the
# annotation dataframe, and so can be manipulated easily.
def annotate_tree():
    tree = load_tree()
    anno = load_annotation()
    
    leaves = []  # Will hold leaves for the tree where they match the annotation row, or None if there is none
    
    for id in anno["id"]:               # iterate over annotations
        for leaf in tree.iter_leaves():  # iterate over all leaves in the tree
            assigned = False
            if id in str(leaf):
                leaves.append(leaf)
                assigned = True
                break
        if not assigned:
            leaves.append(None)
    
    anno["leaves"] = leaves

    return tree, anno

In [6]:
tree, anno = annotate_tree()  # get the annotated tree
anno

,id,kingdom,match,leaves
0,A0A556RIY8,Bacteria,NaN,(((\n--Gilliamella apicola Nitrogen regulatory...
1,K8W564,Bacteria,NaN,(((\n--Providencia sneebia DSM 19967 Nitrogen ...
2,A0AA87CRL5,Bacteria,NaN,(((\n--Providencia stuartii ATCC 25827 Nitroge...
3,B6XEC6,Bacteria,NaN,(((\n--Providencia alcalifaciens DSM 30120 Nit...
4,K8X885,Bacteria,NaN,(((\n--Providencia burhodogranariea DSM 19968 ...
...,...,...,...,...
1671,A0A7R6P982,Bacteria,NaN,(((\n--Neptunomonas japonica JAMM 1380 Nitroge...
1672,A0A7R6PDR3,Bacteria,NaN,(((\n--Neptunomonas japonica JAMM 1380 Nitroge...
1673,A0A7U8C906,Bacteria,NaN,(((\n--Neptuniibacter caesariensis Regulatory ...
1674,A0A2G6JCW0,Bacteria,NaN,(((\n--Neptuniibacter caesariensis Transcripti...


In [26]:
tree, anno = annotate_tree()  # get a clean tree

# Declare a style (we could have put this in a function to save repeated code)
kingdom = TreeStyle()
kingdom.show_leaf_name = True
kingdom.mode = "c"
kingdom.show_branch_support = True

tree.ladderize()
R = tree.get_midpoint_outgroup()
tree.set_outgroup(R)

# One text face for each kingdom
face_dict = {"Eukaryota": TextFace("eukaryota"),
             "Bacteria": TextFace("bacteria"),
             "Archaea": TextFace("archaea")}

# Set colours for kingdoms
colour_dict = {"Eukarya": "#FFFACD", "Bacteria": "#F0F8FF", "Archaea": "#FFE4E1"}

# Iterate over all annotated leaves and add the face
for idx, row in anno.iterrows():
    # Get annotation information
    leaf = row["leaves"]
    kingdomname = row["kingdom"]


    # Set styling for the leaf node
    # leaf.img_style["size"] = 12
    if kingdomname in ("Eukarya", "Bacteria", "Archaea"):
        leaf.img_style["bgcolor"] = colour_dict[kingdomname]    
        leaf.add_face(face_dict[kingdomname], 0, "aligned")

tree.render("figure_3.6.pdf", tree_style=kingdom, w=24, h=24, units="in");

In [5]:
my_file = open("domain_info.txt", "r") 
  
# reading the file 
dat = my_file.read() 
  
# replacing end splitting the text  
# when newline ('\n') is seen. 
domain_info = dat.split("\n") 
print(domain_info) 
my_file.close() 

["Q8GQS1: Ammonium Transporter Family [{'start': 50, 'end': 460, 'dc-status': 'CONTINUOUS'}] 463", "A0A401X433: Ammonium Transporter Family [{'start': 51, 'end': 461, 'dc-status': 'CONTINUOUS'}] 465", "A0AAC9SUX5: Ammonium Transporter Family [{'start': 45, 'end': 455, 'dc-status': 'CONTINUOUS'}] 459", "F1YU33: Ammonium Transporter Family [{'start': 52, 'end': 462, 'dc-status': 'CONTINUOUS'}] 466", "A0A1D8QW75: Ammonium Transporter Family [{'start': 52, 'end': 462, 'dc-status': 'CONTINUOUS'}] 466", "A0A401WUW1: Ammonium Transporter Family [{'start': 52, 'end': 462, 'dc-status': 'CONTINUOUS'}] 466", "A0A1Y0UZX2: Ammonium Transporter Family [{'start': 43, 'end': 453, 'dc-status': 'CONTINUOUS'}] 457", "A0A291PMC6: Ammonium Transporter Family [{'start': 37, 'end': 447, 'dc-status': 'CONTINUOUS'}] 450", "A0A0U5EV31: Ammonium Transporter Family [{'start': 51, 'end': 461, 'dc-status': 'CONTINUOUS'}] 464", "A0A149U2G9: Ammonium Transporter Family [{'start': 51, 'end': 461, 'dc-status': 'CONTINU

In [6]:
# Parse the data from the domain info file into a dictionary keyed by accession number
domain_dict = defaultdict(list)

with Path("domain_info.txt").open() as ifh:    # Using a file context so that everything is closed when we're done
    for line in [_ for _ in ifh.readlines()]:  # Iterating over each line
        accn, data = line.split(": ", 1)       # Get the accession to use as a key, and the data for processing
        loc, name = data.split("]")            # Get the domain name for the text label
        namedata = name.strip().split()        # Split domain name and sequence length
        label, length = " ".join(namedata[:-1]), int(namedata[-1])
        locdata = loc.split(": ")
        start, end = int(locdata[1].split(",")[0]), int(locdata[2].split(",")[0])
        domain_dict[accn].append((start, end, label, length))  # Adds the current motif to a list of motifs

# Visualise the first few lines of content
print(f"{list(domain_dict.items())[:3]=}\n")

# Visualise a multidomain entry
print(f'{domain_dict["Q1Q357"]=}\n')

list(domain_dict.items())[:3]=[('Q8GQS1', [(50, 460, '', 463)]), ('A0A401X433', [(51, 461, '', 465)]), ('A0AAC9SUX5', [(45, 455, '', 459)])]

domain_dict["Q1Q357"]=[(451, 518, '', 679), (11, 408, '', 679), (565, 674, '', 679)]



In [16]:
def result_to_face(domains, fontsize=2):
    """Return a SeqMotifFace, based on the passed UniProt result
    
    LP: We need to make a change here because the result received by the function
        is not the same as in the previous notebook. It's a list of
        (start, end, label, legnth) tuples
    """
    # LP: We can lose most of the prep work here as we did it on parsing
    motifs = []
    for start, end, label, length in domains:
        # seq.start, seq.end, shape, width, height, fgcolor, bgcolor
        if "Ammonium Transporter" in label:   # LP: change here for how we check for a transporter domain
            motifs.append([start, end, "[]", None, 10, "lightblue", "lightblue", f"arial|{fontsize}|white|{label}"])
        else:
            motifs.append([start, end, "[]", None, 10, "black", "red", f"arial|{fontsize}|white|{label}"])
    if len(motifs):
        if length is None:
            return SeqMotifFace(seq=None, motifs=motifs, seq_format="-")
        else:
            return SeqMotifFace(seq="-" * length, motifs=motifs, seq_format="-")
    return None

In [17]:
tree, anno = annotate_tree()  # get clean tree

# Declare tree style
everything = TreeStyle()
everything.show_leaf_name = False
everything.mode = "c"

# One text face for each kingdom
face_dict = {"Eukaryota": TextFace("eukaryota"),
             "Bacteria": TextFace("bacteria"),
             "Archaea": TextFace("archaea")}

# Set colours for kingdoms
colour_dict = {"Eukaryota": "#FFFACD", "Bacteria": "#F0F8FF", "Archaea": "#FFE4E1"}

# Set face colours for kingdoms
colour_dict = {"Eukaryota": "#FFFACD", "Bacteria": "#F0F8FF", "Archaea": "#FFE4E1"}

# Set colours for residue types
# rescolours = {"Y": "#FFCC66", "E": "#009933", "M": "#9966FF"}

# Iterate over leaves
for idx, row in tqdm(anno.iterrows()):
    # Get leaf information
    leaf = row["leaves"]
    # restype = row["first"]
    # fourres = row["four"]    
    kingdomname = row["kingdom"].strip()    
    accn = str(leaf).split()[-1]     # LP: Use the accession to key/query the domain dictionary

    # # Style the leaf node
    # leaf.img_style["bgcolor"] = colour_dict[kingdomname]    
    # leaf.add_face(face_dict[kingdomname], 1, "aligned")

    # # Add gateway residue bubble
    # if restype in ("Y", "E", "M"):
    #     face = CircleFace(radius=20, color=rescolours[restype], style="sphere", label=restype)
    #     face.opacity = 0.3
    #     leaf.add_face(face, 1, position="float")

    # # Add four residues label
    # if len(fourres):
    #     leaf.add_face(TextFace(fourres), 2, position="float")

    # Add motifs/domains
    # LP: Our first change is here - we want to use the leaf's accession number to
    #     query domain_dict
    motifs = result_to_face(domain_dict[accn])
    if motifs is not None:
        leaf.add_face(motifs, 0, "aligned")

tree.render("figure_4.2.pdf", tree_style=everything, w=24, h=24, units="in");

0it [00:00, ?it/s]

In [13]:
tree, anno = annotate_tree()  # get clean tree

# Declare tree style
everything = TreeStyle()
everything.show_leaf_name = False
everything.mode = "c"
tree.show_branch_support = True


tree.ladderize()

face_dict = {"Eukaryota": TextFace("eukaryota"),
             "Bacteria": TextFace("bacteria"),
             "Archaea": TextFace("archaea")}

# Set face colours for kingdoms
colour_dict = {"Eukaryota": "ForestGreen", "Bacteria": "Tan", "Archaea": "IndianRed"}

# Define function symbols
symbols = {"yes": "<    >"}

# Iterate over leaves
for idx, row in anno.iterrows():
    # Get leaf information
    leaf = row["leaves"]
    fused = row["match"]    
    kingdomname = row["kingdom"]   

    # Style the leaf node
    leaf.img_style["bgcolor"] = colour_dict[kingdomname]    
    leaf.add_face(face_dict[kingdomname], 1, "aligned")

    # Add function symbol
    if fused in symbols:
        leaf.add_face(TextFace(symbols[fused], fsize=50), 2, position="float")

tree.render("figure_4.5.pdf", tree_style=everything, w=24, h=24, units="in");